In [1]:
!pip install dagshub mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import dagshub
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("XGBoost")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())
print("✅ XGBoost version:", xgb.__version__)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=94b8988c-7a79-490b-abcf-2729c10e700a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4d97491e9d479283a5c582766c6835e470b36ba8579b3be608f7f5d30692d139




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow
✅ XGBoost version: 3.2.0


# **Cleaning**

In [3]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD   = 0.80
    ID_MISSING_THRESHOLD    = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns  = [c.replace('-', '_') for c in test_id.columns]
    train_id.columns = [c.replace('-', '_') for c in train_id.columns]

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    y_train_xgb = train["isFraud"].copy()
    X_train_xgb = train.drop(columns=["isFraud", "TransactionID"])
    X_test_xgb  = test.drop(columns=["TransactionID"])

    del train, test, train_txn, train_id, test_txn, test_id
    gc.collect()

    id_like_cols  = [c for c in X_train_xgb.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train_xgb.columns if c not in id_like_cols]

    missing_ratio = X_train_xgb.isnull().mean()
    drop_txn      = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id       = [c for c in id_like_cols  if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing  = drop_txn + drop_id

    X_train_xgb = X_train_xgb.drop(columns=drop_missing)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in drop_missing if c in X_test_xgb.columns])

    near_constant_cols = []
    for col in X_train_xgb.columns:
        top_freq = X_train_xgb[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train_xgb = X_train_xgb.drop(columns=near_constant_cols)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in near_constant_cols if c in X_test_xgb.columns])

    for col in X_train_xgb.columns:
        if col not in X_test_xgb.columns:
            X_test_xgb[col] = np.nan
    X_test_xgb = X_test_xgb[X_train_xgb.columns]

    mlflow.log_param("txn_missing_threshold",   TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold",    ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)
    mlflow.log_metric("train_rows",             int(X_train_xgb.shape[0]))
    mlflow.log_metric("test_rows",              int(X_test_xgb.shape[0]))
    mlflow.log_metric("features_after_cleaning",int(X_train_xgb.shape[1]))
    mlflow.log_metric("dropped_high_missing",   len(drop_missing))
    mlflow.log_metric("dropped_near_constant",  len(near_constant_cols))
    mlflow.log_metric("fraud_rate",             float(y_train_xgb.mean()))

    print(f"X_train: {X_train_xgb.shape}")
    print(f"X_test:  {X_test_xgb.shape}")
    print(f"Fraud rate: {y_train_xgb.mean():.4f}")

    X_train_clean_xgb = X_train_xgb
    X_test_clean_xgb  = X_test_xgb
    y_train_clean_xgb = y_train_xgb

X_train: (590540, 353)
X_test:  (506691, 353)
Fraud rate: 0.0350
🏃 View run XGBoost_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/fceaad1dee1b4e0093dd1e13d1d32ade
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Engineering

In [4]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    X_train = X_train_clean_xgb.copy()
    X_test  = X_test_clean_xgb.copy()
    y_train = y_train_clean_xgb.copy()

    # log transform — TransactionAmt is heavily right-skewed
    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"]  = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    # cyclic hour encoding — fraud has strong time-of-day pattern
    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"]  = np.sin(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)
    X_test["hour_cos"]  = np.cos(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test  = X_test.drop(columns=["TransactionDT"],  errors="ignore")

    # ordinal encode categoricals
    # XGBoost handles ordinal-encoded cats fine — no one-hot needed
    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_param("cat_encoding",     "ordinal_train_mapping")
    mlflow.log_param("amt_log_transform", True)
    mlflow.log_param("cyclic_time_encoding", True)
    mlflow.log_metric("features_after_fe",   int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded",    len(cat_cols))

    print(f"X_train_fe: {X_train.shape}")
    print(f"X_test_fe:  {X_test.shape}")

    X_train_fe_xgb = X_train
    X_test_fe_xgb  = X_test
    y_train_fe_xgb = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run XGBoost_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/cb5ae353f15b457d9d2d3647bb1e3d1c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Selection

In [5]:
with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    X_train = X_train_fe_xgb.copy()
    X_test  = X_test_fe_xgb.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    # drop constant columns
    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test  = X_test.drop(columns=const_cols,  errors="ignore")

    # drop highly correlated features — sampled for speed
    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    sample_n = min(120_000, len(X_train))
    idx  = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test  = X_test.drop(columns=drop_corr,  errors="ignore")
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("corr_threshold",     CORR_THRESHOLD)
    mlflow.log_metric("dropped_const",     len(const_cols))
    mlflow.log_metric("dropped_high_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print(f"X_train_fs: {X_train.shape}")
    print(f"X_test_fs:  {X_test.shape}")

    X_train_final_xgb = X_train
    X_test_final_xgb  = X_test

X_train_fs: (590540, 301)
X_test_fs:  (506691, 301)
🏃 View run XGBoost_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/fa203c0afa6b45fb8e316b57c12526be
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Training

In [6]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.2, random_state=42, stratify=y_train_fe_xgb
)

neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
spw = round(neg / pos, 2)
print(f"Train size: {X_tr.shape} | Val size: {X_val.shape}")
print(f"scale_pos_weight = {spw}  (neg/pos = {neg}/{pos})")

Train size: (472432, 301) | Val size: (118108, 301)
scale_pos_weight = 27.58  (neg/pos = 455902/16530)


In [7]:
with mlflow.start_run(run_name="XGB_Baseline"):
    mlflow.log_param("n_estimators",     100)
    mlflow.log_param("max_depth",        6)
    mlflow.log_param("learning_rate",    0.3)
    mlflow.log_param("scale_pos_weight", 1)
    mlflow.log_param("tree_method",      "hist")
    mlflow.log_param("device",           "cuda")
    mlflow.log_param("note",             "default params, no class balancing")

    clf = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.3,
        scale_pos_weight=1, tree_method="hist", device="cuda",
        eval_metric="auc", random_state=42
    )
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(gap, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print("  -> default lr=0.3, no class balancing.")
    print(f"  -> Gap {gap:.4f}: {'OVERFIT' if gap > 0.02 else 'acceptable'}")

[Baseline] Train: 0.9563 | Val: 0.9388 | Gap: 0.0175
  -> default lr=0.3, no class balancing.
  -> Gap 0.0175: acceptable
🏃 View run XGB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/9180643f3c0d4172aab699fddcf4c884
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [8]:
for n_est in [100, 300, 500]:
    with mlflow.start_run(run_name=f"XGB_nestimators_{n_est}"):
        mlflow.log_param("n_estimators",     n_est)
        mlflow.log_param("max_depth",        6)
        mlflow.log_param("learning_rate",    0.1)
        mlflow.log_param("scale_pos_weight", spw)
        mlflow.log_param("tree_method",      "hist")
        mlflow.log_param("device",           "cuda")

        clf = xgb.XGBClassifier(
            n_estimators=n_est, 
            max_depth=6, 
            learning_rate=0.1,
            scale_pos_weight=spw, 
            tree_method="hist", 
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50 if n_est > 100 else None, 
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        status = "OVERFIT" if gap > 0.025 else ("UNDERFIT" if val_auc < 0.90 else "OK")
        print(f"  n_estimators={n_est:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f} [{status}]")

  n_estimators=100  | Train: 0.9370 | Val: 0.9227 | Gap: 0.0143 [OK]
🏃 View run XGB_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/264679de82f14f7d86787f38646df17b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_estimators=300  | Train: 0.9686 | Val: 0.9458 | Gap: 0.0228 [OK]
🏃 View run XGB_nestimators_300 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/c0494e108b42405eb94e82d1226537b8
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_estimators=500  | Train: 0.9809 | Val: 0.9537 | Gap: 0.0272 [OVERFIT]
🏃 View run XGB_nestimators_500 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/6f8b65962cc9417ea5b968e93ebfca8b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [9]:
depth_results = []
for depth in [3, 4, 6, 8, 10]:
    with mlflow.start_run(run_name=f"XGB_depth_{depth}"):
        # პარამეტრების ლოგირება
        mlflow.log_params({
            "n_estimators": 300,
            "max_depth": depth,
            "learning_rate": 0.1,
            "scale_pos_weight": spw,
            "tree_method": "hist",
            "device": "cuda",
            "early_stopping_rounds": 50
        })

        clf = xgb.XGBClassifier(
            n_estimators=300, 
            max_depth=depth, 
            learning_rate=0.1,
            scale_pos_weight=spw, 
            tree_method="hist", 
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50, # დავამატოთ აქაც, რომ ციკლში სტაბილური იყოს
            random_state=42
        )
        
        # Fit მეთოდი ვალიდაციის სეტით
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        # პროგნოზისთვის ვიყენებთ საუკეთესო იტერაციას (best_iteration)
        best_iter = clf.best_iteration
        y_tr_pred = clf.predict_proba(X_tr, iteration_range=(0, best_iter + 1))[:, 1]
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]

        train_auc = roc_auc_score(y_tr, y_tr_pred)
        val_auc   = roc_auc_score(y_val, y_val_pred)
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc", round(train_auc, 5))
        mlflow.log_metric("val_auc",   round(val_auc, 5))
        mlflow.log_metric("overfit_gap", round(gap, 5))
        mlflow.log_metric("best_iteration", best_iter)

        if gap > 0.035: 
            diagnosis = "OVERFIT — memorising train"
        elif val_auc < 0.91:
            diagnosis = "UNDERFIT — lacking capacity"
        else:
            diagnosis = "OK"

        depth_results.append({
            "max_depth": depth, 
            "train_auc": train_auc,
            "val_auc": val_auc, 
            "gap": gap,
            "best_iter": best_iter
        })
        
        print(f"  depth={depth:<2} | Best Iter: {best_iter:<3} | Val AUC: {val_auc:.4f} | Gap: {gap:.4f} — {diagnosis}")

depth_df = pd.DataFrame(depth_results).set_index("max_depth")
best_depth = int(depth_df["val_auc"].idxmax())
print(f"\n✅ Best depth found: {best_depth}")

  depth=3  | Best Iter: 299 | Val AUC: 0.9076 | Gap: 0.0086 — UNDERFIT — lacking capacity
🏃 View run XGB_depth_3 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/9b553bb152004c1297e0a9d50caa5906
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=4  | Best Iter: 299 | Val AUC: 0.9221 | Gap: 0.0128 — OK
🏃 View run XGB_depth_4 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/4380f79a21a64474b30cec3d729bb1a6
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=6  | Best Iter: 299 | Val AUC: 0.9458 | Gap: 0.0228 — OK
🏃 View run XGB_depth_6 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/18b4670c1c4a4666aba19125436dd820
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth=8  | Best Iter: 299 | Val AUC: 0.9582 | Gap

In [10]:
sampling_results = []
for subsample, colsample in [(1.0, 1.0), (0.8, 0.8), (0.7, 0.7)]:
    label = f"sub{subsample}_col{colsample}"
    with mlflow.start_run(run_name=f"XGB_sampling_{label}"):
        clf = xgb.XGBClassifier(
            n_estimators=300, 
            max_depth=best_depth, 
            learning_rate=0.1,
            subsample=subsample, 
            colsample_bytree=colsample,
            scale_pos_weight=spw, 
            tree_method="hist", 
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        best_iter = clf.best_iteration
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred = clf.predict_proba(X_tr, iteration_range=(0, best_iter + 1))[:, 1]

        train_auc = roc_auc_score(y_tr, y_tr_pred)
        val_auc = roc_auc_score(y_val, y_val_pred)
        gap = train_auc - val_auc

        mlflow.log_params({"subsample": subsample, "colsample_bytree": colsample})
        mlflow.log_metrics({"train_auc": train_auc, "val_auc": val_auc, "gap": gap})

        sampling_results.append({"label": label, "sub": subsample, "col": colsample, "val_auc": val_auc})
        print(f"  {label:<15} | Best Iter: {best_iter:<3} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

best_sampling = pd.DataFrame(sampling_results).loc[pd.DataFrame(sampling_results)["val_auc"].idxmax()]
best_subsample = best_sampling["sub"]
best_colsample = best_sampling["col"]

  sub1.0_col1.0   | Best Iter: 299 | Val: 0.9636 | Gap: 0.0338
🏃 View run XGB_sampling_sub1.0_col1.0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/a60fddc4b2384939abb0994a9c93fb92
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.8_col0.8   | Best Iter: 299 | Val: 0.9657 | Gap: 0.0338
🏃 View run XGB_sampling_sub0.8_col0.8 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/eb5b115b144c41bb8150be1b21be1403
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.7_col0.7   | Best Iter: 299 | Val: 0.9655 | Gap: 0.0341
🏃 View run XGB_sampling_sub0.7_col0.7 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/31fe01c33f704edbae8c2847b99cae28
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [11]:
reg_results = []
for alpha, lam in [(0, 1), (0.1, 1), (1, 1), (0, 5), (1, 5)]:
    label = f"a{alpha}_l{lam}"
    with mlflow.start_run(run_name=f"XGB_reg_{label}"):
        clf = xgb.XGBClassifier(
            n_estimators=300, 
            max_depth=best_depth, 
            learning_rate=0.1,
            subsample=best_subsample, 
            colsample_bytree=best_colsample,
            reg_alpha=alpha, 
            reg_lambda=lam,
            scale_pos_weight=spw, 
            tree_method="hist", 
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        best_iter = clf.best_iteration
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred = clf.predict_proba(X_tr, iteration_range=(0, best_iter + 1))[:, 1]

        train_auc = roc_auc_score(y_tr, y_tr_pred)
        val_auc = roc_auc_score(y_val, y_val_pred)
        gap = train_auc - val_auc

        mlflow.log_metrics({"train_auc": train_auc, "val_auc": val_auc, "gap": gap})

        reg_results.append({"alpha": alpha, "lambda": lam, "val_auc": val_auc, "gap": gap})
        print(f"  alpha={alpha} lambda={lam} | Best Iter: {best_iter:<3} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

reg_df = pd.DataFrame(reg_results)
best_reg = reg_df.loc[reg_df["val_auc"].idxmax()]
best_alpha, best_lambda = best_reg["alpha"], best_reg["lambda"]

  alpha=0 lambda=1 | Best Iter: 299 | Val: 0.9657 | Gap: 0.0338
🏃 View run XGB_reg_a0_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/342a98119c354854ba5fe6306c1db759
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=0.1 lambda=1 | Best Iter: 299 | Val: 0.9658 | Gap: 0.0337
🏃 View run XGB_reg_a0.1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/e2a9fd9640f944d5907ee93dd416e565
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=1 lambda=1 | Best Iter: 299 | Val: 0.9663 | Gap: 0.0333
🏃 View run XGB_reg_a1_l1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/d5ed845b79d74646b5d0896e21c5734f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  alpha=0 lambda=5 | Best Iter: 294 | Val: 0.9665 | Gap: 0.0326
🏃 Vie

In [12]:
lr_results = []
for lr in [0.01, 0.05, 0.1, 0.2]:
    with mlflow.start_run(run_name=f"XGB_lr_{lr}"):
        clf = xgb.XGBClassifier(
            n_estimators=300 if lr >= 0.05 else 600, # დაბალ LR-ზე მეტი ხე
            max_depth=best_depth, 
            learning_rate=lr,
            subsample=best_subsample, 
            colsample_bytree=best_colsample,
            reg_alpha=best_alpha, 
            reg_lambda=best_lambda,
            scale_pos_weight=spw, 
            tree_method="hist", 
            device="cuda",
            eval_metric="auc",
            early_stopping_rounds=50,
            random_state=42
        )
        
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        best_iter = clf.best_iteration
        y_val_pred = clf.predict_proba(X_val, iteration_range=(0, best_iter + 1))[:, 1]
        y_tr_pred = clf.predict_proba(X_tr, iteration_range=(0, best_iter + 1))[:, 1]

        train_auc = roc_auc_score(y_tr, y_tr_pred)
        val_auc = roc_auc_score(y_val, y_val_pred)
        gap = train_auc - val_auc

        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("best_iteration", best_iter)

        lr_results.append({"lr": lr, "train_auc": train_auc, "val_auc": val_auc, "gap": gap, "best_iter": best_iter})
        print(f"  lr={lr:<5} | Best Iter: {best_iter:<3} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

lr_df = pd.DataFrame(lr_results).set_index("lr")
best_lr = float(lr_df["val_auc"].idxmax())

if best_lr <= 0.01:
    best_n_est = 2000
elif best_lr <= 0.05:
    best_n_est = 1000
else:
    best_n_est = 500

print(f"\n✅ Best LR: {best_lr} | Suggested Final N: {best_n_est}")

  lr=0.01  | Best Iter: 599 | Val: 0.9468 | Gap: 0.0280
🏃 View run XGB_lr_0.01 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/d791aa9bd4fc4e628a0078104c1ee8d1
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  lr=0.05  | Best Iter: 299 | Val: 0.9627 | Gap: 0.0314
🏃 View run XGB_lr_0.05 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/59656fbaaa3b4108aec31304bb51cc34
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  lr=0.1   | Best Iter: 298 | Val: 0.9667 | Gap: 0.0326
🏃 View run XGB_lr_0.1 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/82ef335de0c443359c89a3efafaa59c4
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  lr=0.2   | Best Iter: 299 | Val: 0.9680 | Gap: 0.0320
🏃 View run XGB_lr_0.2 at: https://dagshub.com/so

In [13]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

sample_idx = np.random.RandomState(42).choice(len(X_train_final_xgb), size=150_000, replace=False)
X_cv = X_train_final_xgb.iloc[sample_idx]
y_cv = y_train_fe_xgb.iloc[sample_idx]

with mlflow.start_run(run_name="XGB_CrossValidation_Final"):
    params = {
        "n_estimators": int(best_n_est),
        "max_depth": int(best_depth),
        "learning_rate": float(best_lr),
        "subsample": float(best_subsample),
        "colsample_bytree": float(best_colsample),
        "reg_alpha": float(best_alpha),
        "reg_lambda": float(best_lambda),
        "scale_pos_weight": float(spw),
        "tree_method": "hist",
        "device": "cuda",  
        "random_state": 42
    }
    
    mlflow.log_params(params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    print(f"🚀 Starting CV on T4 GPU (Best N: {best_n_est}, Best LR: {best_lr})")
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cv, y_cv)):
        X_tr_fold, X_val_fold = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
        y_tr_fold, y_val_fold = y_cv.iloc[train_idx], y_cv.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params, eval_metric="auc")
        
        model.fit(X_tr_fold, y_tr_fold)
        
        score = roc_auc_score(y_val_fold, model.predict_proba(X_val_fold)[:, 1])
        cv_scores.append(score)
        
        mlflow.log_metric("fold_auc", round(score, 5), step=fold)
        print(f"  Fold {fold+1}: AUC = {score:.4f}")

    avg_auc = np.mean(cv_scores)
    std_auc = np.std(cv_scores)
    
    mlflow.log_metrics({"cv_mean": avg_auc, "cv_std": std_auc})
    
    print(f"\n✅ Mean AUC: {avg_auc:.4f} (+/- {std_auc:.4f})")
    if std_auc < 0.005:
        print("💎 Model is very stable!")

🚀 Starting CV on T4 GPU (Best N: 500, Best LR: 0.2)
  Fold 1: AUC = 0.9354
  Fold 2: AUC = 0.9377
  Fold 3: AUC = 0.9331
  Fold 4: AUC = 0.9416
  Fold 5: AUC = 0.9277

✅ Mean AUC: 0.9351 (+/- 0.0046)
💎 Model is very stable!
🏃 View run XGB_CrossValidation_Final at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/9e44320a2b984f849b9d5ebafefdfae4
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [15]:
import pickle
from sklearn.pipeline import Pipeline

X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.2, random_state=42, stratify=y_train_fe_xgb
)

neg_f = int((y_tr_f == 0).sum())
pos_f = int((y_tr_f == 1).sum())
spw_f = round(neg_f / pos_f, 2)

with mlflow.start_run(run_name="XGB_Final_Pipeline") as run:
    params = {
        "n_estimators": int(best_n_est),
        "max_depth": int(best_depth),
        "learning_rate": float(best_lr),
        "subsample": float(best_subsample),
        "colsample_bytree": float(best_colsample),
        "reg_alpha": float(best_alpha),
        "reg_lambda": float(best_lambda),
        "scale_pos_weight": spw_f,
        "tree_method": "hist",
        "device": "cuda",
        "random_state": 42
    }
    mlflow.log_params(params)
    mlflow.log_param("note", "Fully dynamic pipeline based on all optimization sweeps")

    final_clf = xgb.XGBClassifier(**params, eval_metric="auc", early_stopping_rounds=50)
    
    final_clf.fit(X_tr_f, y_tr_f, eval_set=[(X_val_f, y_val_f)], verbose=False)

    best_iter = final_clf.best_iteration
    y_val_pred = final_clf.predict_proba(X_val_f, iteration_range=(0, best_iter + 1))[:, 1]
    y_tr_pred = final_clf.predict_proba(X_tr_f, iteration_range=(0, best_iter + 1))[:, 1]

    val_auc = roc_auc_score(y_val_f, y_val_pred)
    train_auc = roc_auc_score(y_tr_f, y_tr_pred)
    gap = train_auc - val_auc

    mlflow.log_metrics({
        "train_auc": round(train_auc, 5),
        "val_auc": round(val_auc, 5),
        "overfit_gap": round(gap, 5),
        "actual_best_iteration": best_iter
    })

    final_pipe = Pipeline([("clf", final_clf)])

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="xgboost_pipeline",
        registered_model_name="XGBoost_FraudDetection"
    )

    pkl_path = "xgboost_final_pipeline.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_pipe, f)
    mlflow.log_artifact(pkl_path)

    print(f"🚀 Final Model Training Complete!")
    print(f"Params used: LR={params['learning_rate']}, Max Depth={params['max_depth']}")
    print(f"Results -> Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")
    print(f"Run ID: {run.info.run_id}")

2026/05/05 11:05:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 11:05:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGBoost_FraudDetection' already exists. Creating a new version of this model...
2026/05/05 11:05:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 5
Created version '5' of model 'XGBoost_FraudDetection'.


🚀 Final Model Training Complete!
Params used: LR=0.2, Max Depth=10
Results -> Train: 1.0000 | Val: 0.9688 | Gap: 0.0312
Run ID: 98c4565bce384c66b3b3ddbdfb8cc80b
🏃 View run XGB_Final_Pipeline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/98c4565bce384c66b3b3ddbdfb8cc80b
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
